# CS7 S03: CPU-GPU Workload Partitioning

In [3]:

STUDENT_ID = "23BCE0500"
BATCH_ID = "CS7"
NAME = "Aakaash SS"
REG = "23BCE0500"
BATCH_SECTION = "CS7"

In [4]:
!nvidia-smi
!nvcc --version
!nproc
!free -g
!pip install -q python-docx

Mon Oct 19 16:07:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile s03_bench.cu
// S03 - CPU-GPU workload partitioning benchmark (CUDA + OpenMP)
//
// Workload: N independent image tiles, each T x T floats. Every tile is processed by P passes of a
// 3x3 box blur (clamped edges inside the tile). Tiles are independent, so any subset can run on
// either device. A fraction f_cpu of the tiles goes to the CPU (OpenMP), the rest to the GPU (CUDA).
//   f_cpu = 1.0 -> CPU-only baseline,  f_cpu = 0.0 -> GPU-only baseline,  otherwise hybrid.
//
// The GPU work (H2D copy, P kernels, D2H copy, in chunks) is enqueued asynchronously first, then the
// host runs its own share with OpenMP while the GPU works, and finally waits for the GPU.
// Raw per-run measurements are printed as CSV rows on stdout, one row per (cpu_frac, rep).
//
// Build (GPU):   nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native s03_bench.cu -o s03_bench
// Build (no GPU, pipeline test only):  g++ -O3 -fopenmp -march=native -DNO_CUDA -x c++ s03_bench.cu -o s03_bench_cpu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cstdint>
#include <cmath>
#include <vector>
#include <string>
#include <algorithm>
#include <omp.h>

#ifndef NO_CUDA
#include <cuda_runtime.h>
#else
// ---- host-only shim: lets the whole pipeline be tested on a machine without a GPU.
// The "GPU" runs synchronously on the host, so timings from this build say nothing about real hardware.
typedef int cudaError_t;
static const cudaError_t cudaSuccess = 0;
typedef int cudaStream_t;
struct ShimEvent { double t; };
typedef ShimEvent* cudaEvent_t;
enum cudaMemcpyKind { cudaMemcpyHostToDevice, cudaMemcpyDeviceToHost };
#define cudaHostAllocDefault 0
static const char* cudaGetErrorString(cudaError_t) { return "no error"; }
static cudaError_t cudaMalloc(void** p, size_t n) { *p = malloc(n); return *p ? 0 : 2; }
static cudaError_t cudaFree(void* p) { free(p); return 0; }
static cudaError_t cudaHostAlloc(void** p, size_t n, unsigned) { *p = malloc(n); return *p ? 0 : 2; }
static cudaError_t cudaFreeHost(void* p) { free(p); return 0; }
static cudaError_t cudaStreamCreate(cudaStream_t* s) { *s = 0; return 0; }
static cudaError_t cudaStreamDestroy(cudaStream_t) { return 0; }
static cudaError_t cudaEventCreate(cudaEvent_t* e) { *e = new ShimEvent{0.0}; return 0; }
static cudaError_t cudaEventDestroy(cudaEvent_t e) { delete e; return 0; }
static cudaError_t cudaEventRecord(cudaEvent_t e, cudaStream_t) { e->t = omp_get_wtime() * 1e3; return 0; }
static cudaError_t cudaEventElapsedTime(float* ms, cudaEvent_t a, cudaEvent_t b) { *ms = (float)(b->t - a->t); return 0; }
static cudaError_t cudaStreamWaitEvent(cudaStream_t, cudaEvent_t, unsigned) { return 0; }
static cudaError_t cudaStreamSynchronize(cudaStream_t) { return 0; }
static cudaError_t cudaDeviceSynchronize() { return 0; }
static cudaError_t cudaGetLastError() { return 0; }
static cudaError_t cudaMemcpyAsync(void* d, const void* s, size_t n, cudaMemcpyKind, cudaStream_t) { memcpy(d, s, n); return 0; }
#endif

#define CK(x) do { cudaError_t e_ = (x); if (e_ != cudaSuccess) { \
    fprintf(stderr, "CUDA error '%s' at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); exit(1); } } while (0)

// ------------------------------------------------------------------ GPU kernel
#ifndef NO_CUDA
// 3x3 box blur on a batch of tiles; blockIdx.z selects the tile. Summation order matches the CPU code.
__global__ void k_blur(const float* __restrict__ in, float* __restrict__ out, int T) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= T || y >= T) return;
    size_t base = (size_t)blockIdx.z * T * T;
    const float* src = in + base;
    float s = 0.f;
    for (int dy = -1; dy <= 1; dy++)
        for (int dx = -1; dx <= 1; dx++) {
            int xx = min(max(x + dx, 0), T - 1), yy = min(max(y + dy, 0), T - 1);
            s += src[(size_t)yy * T + xx];
        }
    out[base + (size_t)y * T + x] = s / 9.f;
}
#endif

// ------------------------------------------------------------------ CPU kernel (same arithmetic order)
static inline void blur_edge_px(const float* r0, const float* r1, const float* r2, float* o, int T, int x) {
    int xm = x > 0 ? x - 1 : 0, xp = x < T - 1 ? x + 1 : T - 1;
    float s = 0.f;
    s += r0[xm]; s += r0[x]; s += r0[xp];
    s += r1[xm]; s += r1[x]; s += r1[xp];
    s += r2[xm]; s += r2[x]; s += r2[xp];
    o[x] = s / 9.f;
}

static void blur_tile_cpu(const float* __restrict__ in, float* __restrict__ out, int T) {
    for (int y = 0; y < T; y++) {
        int ym = y > 0 ? y - 1 : 0, yp = y < T - 1 ? y + 1 : T - 1;
        const float* r0 = in + (size_t)ym * T;
        const float* r1 = in + (size_t)y * T;
        const float* r2 = in + (size_t)yp * T;
        float* o = out + (size_t)y * T;
        blur_edge_px(r0, r1, r2, o, T, 0);
#pragma omp simd
        for (int x = 1; x < T - 1; x++) {
            float s = 0.f;
            s += r0[x - 1]; s += r0[x]; s += r0[x + 1];
            s += r1[x - 1]; s += r1[x]; s += r1[x + 1];
            s += r2[x - 1]; s += r2[x]; s += r2[x + 1];
            o[x] = s / 9.f;
        }
        blur_edge_px(r0, r1, r2, o, T, T - 1);
    }
}

// P passes, ping-pong between `out` and `tmp` so that the final pass lands in `out`.
static void cpu_tile(const float* in, float* out, float* tmp, int T, int P) {
    const float* src = in;
    for (int i = 0; i < P; i++) {
        float* dst = (((P - 1 - i) & 1) == 0) ? out : tmp;
        blur_tile_cpu(src, dst, T);
        src = dst;
    }
}

// Runs tiles [first, first+count) on the CPU with `th` OpenMP threads; per-thread busy time -> busy[id*8].
static int cpu_process(const float* h_in, float* h_out, int first, int count, int T, int P, int th, double* busy) {
    if (count <= 0) return 0;
    size_t tp = (size_t)T * T;
    int team = 0;
#pragma omp parallel num_threads(th)
    {
        static thread_local std::vector<float> tmp;
        if (tmp.size() < tp) tmp.resize(tp);
        int id = omp_get_thread_num();
        double b = 0.0;
#pragma omp single
        team = omp_get_num_threads();
#pragma omp for schedule(dynamic, 1)
        for (int k = 0; k < count; k++) {
            double t0 = omp_get_wtime();
            size_t off = (size_t)(first + k) * tp;
            cpu_tile(h_in + off, h_out + off, tmp.data(), T, P);
            b += omp_get_wtime() - t0;
        }
        busy[id * 8] = b;
    }
    return team;
}

// ------------------------------------------------------------------ GPU pipeline
struct GpuCtx {
    int S = 1, chunk = 64, maxChunks = 0;
    std::vector<cudaStream_t> st;
    std::vector<float*> dA, dB;
    std::vector<cudaEvent_t> ev;      // 4 events per chunk: before H2D, after H2D, after kernels, after D2H
    cudaEvent_t evStart = nullptr;
    std::vector<cudaEvent_t> evEnd;   // one per stream
};

static void launch_blur(const float* in, float* out, int T, int nt, cudaStream_t st) {
#ifndef NO_CUDA
    dim3 b(16, 16), g((T + 15) / 16, (T + 15) / 16, nt);
    k_blur<<<g, b, 0, st>>>(in, out, T);
#else
    (void)st;
    for (int t = 0; t < nt; t++) blur_tile_cpu(in + (size_t)t * T * T, out + (size_t)t * T * T, T);
#endif
}

static void gpu_setup(GpuCtx& g, int S, int chunk, int maxTiles, size_t tp) {
    g.S = S; g.chunk = chunk;
    g.maxChunks = (maxTiles + chunk - 1) / chunk;
    g.st.resize(S); g.dA.resize(S); g.dB.resize(S); g.evEnd.resize(S);
    size_t bytes = (size_t)chunk * tp * sizeof(float);
    for (int s = 0; s < S; s++) {
        CK(cudaStreamCreate(&g.st[s]));
        CK(cudaMalloc((void**)&g.dA[s], bytes));
        CK(cudaMalloc((void**)&g.dB[s], bytes));
        CK(cudaEventCreate(&g.evEnd[s]));
    }
    g.ev.resize((size_t)4 * g.maxChunks);
    for (auto& e : g.ev) CK(cudaEventCreate(&e));
    CK(cudaEventCreate(&g.evStart));
}

static void gpu_teardown(GpuCtx& g) {
    for (int s = 0; s < g.S; s++) {
        CK(cudaFree(g.dA[s])); CK(cudaFree(g.dB[s]));
        CK(cudaStreamDestroy(g.st[s])); CK(cudaEventDestroy(g.evEnd[s]));
    }
    for (auto& e : g.ev) CK(cudaEventDestroy(e));
    CK(cudaEventDestroy(g.evStart));
}

// Enqueue tiles [first, first+count) onto the GPU streams (returns immediately; work runs asynchronously).
static int gpu_enqueue(GpuCtx& g, const float* h_in, float* h_out, int first, int count, int T, int P) {
    size_t tp = (size_t)T * T;
    int nch = (count + g.chunk - 1) / g.chunk;
    CK(cudaEventRecord(g.evStart, g.st[0]));
    for (int s = 1; s < g.S; s++) CK(cudaStreamWaitEvent(g.st[s], g.evStart, 0));
    for (int c = 0; c < nch; c++) {
        int s = c % g.S;
        int t0 = first + c * g.chunk;
        int nt = std::min(g.chunk, first + count - t0);
        size_t off = (size_t)t0 * tp, bytes = (size_t)nt * tp * sizeof(float);
        cudaEvent_t* e = &g.ev[(size_t)4 * c];
        CK(cudaEventRecord(e[0], g.st[s]));
        CK(cudaMemcpyAsync(g.dA[s], h_in + off, bytes, cudaMemcpyHostToDevice, g.st[s]));
        CK(cudaEventRecord(e[1], g.st[s]));
        float *a = g.dA[s], *b = g.dB[s];
        for (int p = 0; p < P; p++) { launch_blur(a, b, T, nt, g.st[s]); std::swap(a, b); }
        CK(cudaGetLastError());
        CK(cudaEventRecord(e[2], g.st[s]));
        CK(cudaMemcpyAsync(h_out + off, a, bytes, cudaMemcpyDeviceToHost, g.st[s]));   // `a` holds the latest output
        CK(cudaEventRecord(e[3], g.st[s]));
    }
    for (int s = 0; s < g.S; s++) CK(cudaEventRecord(g.evEnd[s], g.st[s]));
    return nch;
}

struct GpuTimes { double h2d = 0, kern = 0, d2h = 0, span = 0; };

static GpuTimes gpu_collect(GpuCtx& g, int nch) {
    GpuTimes r;
    float ms;
    for (int c = 0; c < nch; c++) {
        cudaEvent_t* e = &g.ev[(size_t)4 * c];
        CK(cudaEventElapsedTime(&ms, e[0], e[1])); r.h2d += ms;
        CK(cudaEventElapsedTime(&ms, e[1], e[2])); r.kern += ms;
        CK(cudaEventElapsedTime(&ms, e[2], e[3])); r.d2h += ms;
    }
    for (int s = 0; s < g.S; s++) {
        CK(cudaEventElapsedTime(&ms, g.evStart, g.evEnd[s]));
        r.span = std::max(r.span, (double)ms);
    }
    return r;
}

// ------------------------------------------------------------------ helpers
static void init_input(float* p, size_t n) {
#pragma omp parallel for
    for (long long i = 0; i < (long long)n; i++) {
        uint32_t x = (uint32_t)i * 2654435761u + 12345u;
        x ^= x >> 16; x *= 0x7feb352dU; x ^= x >> 15; x *= 0x846ca68bU; x ^= x >> 16;
        p[i] = (x >> 8) / 16777216.0f;
    }
}

static double max_rel_err(const float* a, const float* ref, size_t n) {
    double mx = 0.0, mr = 1e-6;
#pragma omp parallel for reduction(max : mx, mr)
    for (long long i = 0; i < (long long)n; i++) {
        double d = std::fabs((double)a[i] - (double)ref[i]);
        double r = std::fabs((double)ref[i]);
        if (d > mx) mx = d;
        if (r > mr) mr = r;
    }
    return mx / mr;
}

static std::string get_arg(int argc, char** argv, const char* key, const char* def) {
    for (int i = 1; i + 1 < argc; i++) if (!strcmp(argv[i], key)) return argv[i + 1];
    return def;
}

static std::vector<double> parse_list(const std::string& s) {
    std::vector<double> v;
    size_t p = 0;
    while (p < s.size()) {
        size_t q = s.find(',', p);
        if (q == std::string::npos) q = s.size();
        if (q > p) v.push_back(atof(s.substr(p, q - p).c_str()));
        p = q + 1;
    }
    return v;
}

struct RunResult {
    int nC = 0, nG = 0, team = 0;
    double makespan = 0, sched = 0, cpu_wall = 0, cpu_busy = 0, sync_wait = 0;
    GpuTimes g;
};

// One timed execution of the whole workload with CPU share f_cpu.
static RunResult run_once(GpuCtx& g, const float* h_in, float* h_out, int N, int T, int P, int th, double f_cpu) {
    RunResult r;
    r.nC = (int)llround(f_cpu * N);
    r.nC = std::max(0, std::min(N, r.nC));
    r.nG = N - r.nC;                       // GPU takes tiles [0, nG), CPU takes [nG, N)
    std::vector<double> busy((size_t)(th + 1) * 8, 0.0);
    CK(cudaDeviceSynchronize());           // start from an idle GPU (untimed)

    double t0 = omp_get_wtime();
    int nch = 0;
    if (r.nG > 0) nch = gpu_enqueue(g, h_in, h_out, 0, r.nG, T, P);
    double t_enq = omp_get_wtime();
    if (r.nC > 0) r.team = cpu_process(h_in, h_out, r.nG, r.nC, T, P, th, busy.data());
    double t_cpu = omp_get_wtime();
    if (r.nG > 0) for (int s = 0; s < g.S; s++) CK(cudaStreamSynchronize(g.st[s]));
    double t1 = omp_get_wtime();

    r.makespan = (t1 - t0) * 1e3;
    r.sched = (t_enq - t0) * 1e3;
    r.cpu_wall = r.nC > 0 ? (t_cpu - t_enq) * 1e3 : 0.0;
    r.sync_wait = r.nG > 0 ? (t1 - t_cpu) * 1e3 : 0.0;
    for (int i = 0; i < r.team && i < th; i++) r.cpu_busy += busy[i * 8] * 1e3;
    if (r.nG > 0) r.g = gpu_collect(g, nch);
    return r;
}

int main(int argc, char** argv) {
    int N = atoi(get_arg(argc, argv, "--tiles", "256").c_str());
    int T = atoi(get_arg(argc, argv, "--tile", "256").c_str());
    int P = atoi(get_arg(argc, argv, "--passes", "1").c_str());
    int th = atoi(get_arg(argc, argv, "--threads", "4").c_str());
    int S = atoi(get_arg(argc, argv, "--streams", "1").c_str());
    int chunk = atoi(get_arg(argc, argv, "--chunk", "64").c_str());
    int pinned = atoi(get_arg(argc, argv, "--pinned", "1").c_str());
    int reps = atoi(get_arg(argc, argv, "--reps", "5").c_str());
    int header = atoi(get_arg(argc, argv, "--header", "1").c_str());
    std::string workload = get_arg(argc, argv, "--workload", "tileblur");
    std::vector<double> fracs = parse_list(get_arg(argc, argv, "--fracs", "0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1"));
    if (N < 1 || T < 4 || P < 1 || th < 1 || S < 1 || chunk < 1 || reps < 1 || fracs.empty()) {
        fprintf(stderr, "bad arguments\n");
        return 1;
    }
    omp_set_dynamic(0);
    CK(cudaFree(0));                       // force CUDA context creation before any timing

    size_t tp = (size_t)T * T, n_el = (size_t)N * tp, bytes = n_el * sizeof(float);
    float *h_in = nullptr, *h_out = nullptr;
    if (pinned) { CK(cudaHostAlloc((void**)&h_in, bytes, cudaHostAllocDefault)); CK(cudaHostAlloc((void**)&h_out, bytes, cudaHostAllocDefault)); }
    else { h_in = (float*)malloc(bytes); h_out = (float*)malloc(bytes); }
    std::vector<float> ref(n_el);
    if (!h_in || !h_out) { fprintf(stderr, "host allocation failed\n"); return 1; }
    init_input(h_in, n_el);

    int maxth = omp_get_max_threads();
    std::vector<double> busy0((size_t)(std::max(1, maxth) + 1) * 8, 0.0);
    cpu_process(h_in, ref.data(), 0, N, T, P, std::max(1, maxth), busy0.data());   // CPU reference result

    GpuCtx g;
    gpu_setup(g, S, chunk, N, tp);
    double dev_bytes = (double)S * 2 * chunk * tp * sizeof(float);

    if (header)
        printf("workload,tiles,tile_dim,passes,threads_req,threads_actual,streams,chunk_tiles,pinned,cpu_frac,tiles_cpu,tiles_gpu,rep,"
               "makespan_ms,sched_ms,cpu_wall_ms,cpu_busy_sum_ms,gpu_h2d_ms,gpu_kernel_ms,gpu_d2h_ms,gpu_span_ms,sync_wait_ms,"
               "max_rel_err,host_bytes,device_bytes\n");

    // warm-up: GPU clocks, OpenMP thread pool, page faults
    const double warm[3] = {0.0, 0.5, 1.0};
    for (double w : warm) run_once(g, h_in, h_out, N, T, P, th, w);

    for (int rep = 0; rep < reps; rep++) {
        std::vector<double> order = fracs;
        if (rep & 1) std::reverse(order.begin(), order.end());   // alternate sweep direction to spread drift
        for (double f : order) {
            bool check = (rep == 0);
            if (check) memset(h_out, 0, bytes);
            RunResult r = run_once(g, h_in, h_out, N, T, P, th, f);
            char errbuf[32];
            if (check) snprintf(errbuf, sizeof errbuf, "%.3e", max_rel_err(h_out, ref.data(), n_el));
            else snprintf(errbuf, sizeof errbuf, "nan");
            printf("%s,%d,%d,%d,%d,%d,%d,%d,%d,%.4f,%d,%d,%d,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%s,%.0f,%.0f\n",
                   workload.c_str(), N, T, P, th, r.team, S, chunk, pinned, f, r.nC, r.nG, rep + 1,
                   r.makespan, r.sched, r.cpu_wall, r.cpu_busy, r.g.h2d, r.g.kern, r.g.d2h, r.g.span, r.sync_wait,
                   errbuf, 2.0 * bytes, dev_bytes);
            fflush(stdout);
        }
        fprintf(stderr, "[%s N=%d P=%d th=%d] rep %d/%d done\n", workload.c_str(), N, P, th, rep + 1, reps);
    }

    gpu_teardown(g);
    if (pinned) { CK(cudaFreeHost(h_in)); CK(cudaFreeHost(h_out)); } else { free(h_in); free(h_out); }
    return 0;
}

Writing s03_bench.cu


In [6]:
%%writefile run_experiment.py
#!/usr/bin/env python3
"""
S03 (CS7) CPU-GPU workload partitioning: experiment driver.

Compiles s03_bench.cu, records the environment, sweeps
    passes x problem size (tiles) x CPU thread count x CPU share,
repeats every point, and stores one raw row per run in results/s03_raw_runs.csv.
Derived metrics (speedup, efficiency, utilisation) are NOT computed here; analyze.py does that
from the raw rows.

Usage (from the folder containing s03_bench.cu):
    python run_experiment.py --quick          # 2-4 minute sanity run
    python run_experiment.py                  # full experiment (defaults below)
    python run_experiment.py --student-id 23BCE0434 --batch-id B07
"""
import argparse, csv, datetime, io, os, platform, re, shutil, subprocess, sys, time
from pathlib import Path


def sh(cmd, check=False, env=None):
    try:
        cp = subprocess.run(cmd, shell=isinstance(cmd, str), text=True, capture_output=True, env=env)
    except Exception as e:  # tool missing
        return 1, "", str(e)
    if check and cp.returncode != 0:
        print(cp.stdout, cp.stderr, file=sys.stderr)
        raise SystemExit(f"command failed: {cmd}")
    return cp.returncode, cp.stdout.strip(), cp.stderr.strip()


def detect_environment():
    info = {}
    rc, out, _ = sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader")
    if rc == 0 and out:
        first = out.splitlines()[0].split(",")
        info["gpu_name"] = first[0].strip()
        info["gpu_mem"] = first[1].strip() if len(first) > 1 else "?"
        info["driver"] = first[2].strip() if len(first) > 2 else "?"
        cc = first[3].strip() if len(first) > 3 else ""
        info["arch"] = "sm_" + cc.replace(".", "") if re.match(r"^\d+\.\d+$", cc) else "sm_75"
        info["gpu_count"] = len(out.splitlines())
    else:
        info.update(gpu_name="none detected", gpu_mem="?", driver="?", arch="sm_75", gpu_count=0)
    cpu = "unknown"
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                cpu = line.split(":", 1)[1].strip()
                break
    except OSError:
        cpu = platform.processor() or "unknown"
    info["cpu_name"] = cpu
    info["logical_cpus"] = os.cpu_count() or 1
    mem_kb = avail_kb = 0
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                mem_kb = int(line.split()[1])
            if line.startswith("MemAvailable"):
                avail_kb = int(line.split()[1])
    except OSError:
        pass
    info["ram_gb"] = round(mem_kb / 1048576, 1)
    info["ram_avail_gb"] = round(avail_kb / 1048576, 1)
    info["nvcc"] = sh("nvcc --version")[1].replace("\n", " | ")
    info["gcc"] = sh("g++ --version")[1].splitlines()[0] if shutil.which("g++") else "g++ not found"
    info["os"] = platform.platform()
    info["python"] = sys.version.split()[0]
    return info


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tiles", type=int, nargs="+", default=[256, 1024, 4096], help="problem sizes: number of tiles")
    ap.add_argument("--tile", type=int, default=256, help="tile edge length in pixels (floats)")
    ap.add_argument("--passes", type=int, nargs="+", default=[1, 16], help="blur passes per tile (arithmetic intensity)")
    ap.add_argument("--threads", type=int, nargs="+", default=[1, 2, 4], help="OpenMP threads for the CPU share")
    ap.add_argument("--streams", type=int, nargs="+", default=[1], help="CUDA streams (1 = serialised offload)")
    ap.add_argument("--pinned", type=int, nargs="+", default=[1], help="1 = pinned host memory, 0 = pageable")
    ap.add_argument("--chunk", type=int, default=64, help="tiles per transfer chunk")
    ap.add_argument("--reps", type=int, default=7)
    ap.add_argument("--fine", action="store_true", help="CPU share step 0.05 instead of 0.1")
    ap.add_argument("--quick", action="store_true", help="small sanity sweep, do not use for reported results")
    ap.add_argument("--skip-compile", action="store_true")
    ap.add_argument("--cpu-only-build", action="store_true", help="build the host-only shim (pipeline test, no GPU)")
    ap.add_argument("--student-id", default="23BCE0434")
    ap.add_argument("--batch-id", default="BATCH_ID")
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--arch", default="auto")
    args = ap.parse_args()

    if args.quick:
        args.tiles, args.passes, args.threads, args.reps = [64, 256], [1, 8], [1, 4], 3

    root = Path(__file__).resolve().parent
    src = root / "s03_bench.cu"
    exe = root / ("s03_bench_cpu" if args.cpu_only_build else "s03_bench")
    out = Path(args.outdir)
    out.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        raise SystemExit("s03_bench.cu must be in the same folder as this script")

    env_info = detect_environment()
    arch = env_info["arch"] if args.arch == "auto" else args.arch
    threads = [t for t in args.threads if t <= env_info["logical_cpus"]]
    if threads != args.threads:
        print(f"note: dropping thread counts above the {env_info['logical_cpus']} logical CPUs available -> {threads}")
    if not threads:
        threads = [env_info["logical_cpus"]]

    if not args.skip_compile:
        if args.cpu_only_build:
            compile_cmd = ["g++", "-O3", "-fopenmp", "-march=native", "-DNO_CUDA", "-x", "c++", str(src), "-o", str(exe)]
        else:
            compile_cmd = ["nvcc", "-O3", "-std=c++14", f"-arch={arch}", "-Xcompiler", "-fopenmp,-O3,-march=native",
                           str(src), "-o", str(exe)]
        print("+", " ".join(compile_cmd), flush=True)
        sh(compile_cmd, check=True)
    else:
        compile_cmd = ["(skipped; reused existing binary)"]
    if not exe.exists():
        raise SystemExit("binary missing; compile first")

    hardware = f"{env_info['gpu_name']} + {env_info['cpu_name']} ({env_info['logical_cpus']} logical CPUs)"
    if args.cpu_only_build:
        hardware = "NO GPU (host-only shim) + " + env_info["cpu_name"]
    (out / "system_info.txt").write_text(
        "S03 environment record\n"
        f"Date: {datetime.datetime.now().isoformat(timespec='seconds')}\n"
        f"GPU: {env_info['gpu_name']} | memory {env_info['gpu_mem']} | driver {env_info['driver']} | target {arch} | GPUs visible: {env_info['gpu_count']}\n"
        f"CPU: {env_info['cpu_name']} | logical CPUs {env_info['logical_cpus']}\n"
        f"RAM: {env_info['ram_gb']} GB total, {env_info['ram_avail_gb']} GB available at start\n"
        f"OS: {env_info['os']}\nPython: {env_info['python']}\n"
        f"CUDA compiler: {env_info['nvcc']}\nHost compiler: {env_info['gcc']}\n"
        f"Compile command: {' '.join(compile_cmd)}\n"
        f"Runtime: OMP_DYNAMIC=FALSE, CUDA_VISIBLE_DEVICES=0, OMP thread counts {threads}\n"
        f"Sweep: tiles={args.tiles} tile={args.tile} passes={args.passes} streams={args.streams} pinned={args.pinned} "
        f"chunk={args.chunk} reps={args.reps} fine={args.fine} quick={args.quick}\n"
    )

    step = 0.05 if args.fine else 0.1
    nfrac = int(round(1 / step))
    fracs = ",".join(f"{i * step:.2f}" for i in range(nfrac + 1))

    run_env = os.environ.copy()
    run_env["OMP_DYNAMIC"] = "FALSE"
    run_env.setdefault("CUDA_VISIBLE_DEVICES", "0")

    configs = [(p, n, th, st, pin) for p in args.passes for n in args.tiles for th in threads
               for st in args.streams for pin in args.pinned]
    raw_path = out / "s03_raw_runs.csv"
    if raw_path.exists():
        raw_path.rename(out / f"s03_raw_runs_previous_{int(time.time())}.csv")
    cols = None
    t_start = time.time()
    run_counter = 0
    print(f"{len(configs)} configurations x {nfrac + 1} CPU shares x {args.reps} repetitions", flush=True)

    for ci, (p, n, th, st, pin) in enumerate(configs, 1):
        need_gb = 3.2 * n * args.tile * args.tile * 4 / 1e9
        if env_info["ram_avail_gb"] and need_gb > env_info["ram_avail_gb"] * 0.8:
            print(f"skip tiles={n}: needs about {need_gb:.1f} GB host RAM, only {env_info['ram_avail_gb']} GB available")
            continue
        cmd = [str(exe), "--tiles", str(n), "--tile", str(args.tile), "--passes", str(p), "--threads", str(th),
               "--streams", str(st), "--chunk", str(args.chunk), "--pinned", str(pin), "--reps", str(args.reps),
               "--fracs", fracs, "--workload", "tileblur"]
        t0 = time.time()
        cp = subprocess.run(cmd, text=True, capture_output=True, env=run_env)
        if cp.returncode != 0:
            print(cp.stdout[-2000:], cp.stderr[-2000:], file=sys.stderr)
            raise SystemExit(f"benchmark failed for {cmd}")
        rows = list(csv.DictReader(io.StringIO(cp.stdout)))
        cfg_id = f"p{p}_n{n}_t{th}_s{st}_pin{pin}"
        for r in rows:
            run_counter += 1
            r["Student_ID"] = args.student_id
            r["Batch_ID"] = args.batch_id
            r["Dataset"] = f"synthetic_tiles_{args.tile}x{args.tile}_float32"
            r["Hardware"] = hardware
            r["Run_ID"] = f"{cfg_id}_f{float(r['cpu_frac']):.2f}_r{r['rep']}"
        if cols is None:
            cols = ["Run_ID", "Student_ID", "Batch_ID", "Dataset", "Hardware"] + [c for c in rows[0].keys()
                                                                                    if c not in ("Run_ID", "Student_ID", "Batch_ID", "Dataset", "Hardware")]
        with open(raw_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            if f.tell() == 0:
                w.writeheader()
            w.writerows(rows)
        bad = [r for r in rows if r["max_rel_err"] != "nan" and float(r["max_rel_err"]) > 1e-5]
        dt = time.time() - t0
        done = time.time() - t_start
        eta = done / ci * (len(configs) - ci)
        flag = f"  WARNING: {len(bad)} runs failed the correctness check" if bad else ""
        print(f"[{ci}/{len(configs)}] {cfg_id}: {len(rows)} rows in {dt:.0f}s, elapsed {done / 60:.1f} min, ETA {eta / 60:.1f} min{flag}", flush=True)

    print(f"\nRaw rows -> {raw_path} ({run_counter} runs)\nEnvironment -> {out / 'system_info.txt'}\nNext: python analyze.py")


if __name__ == "__main__":
    main()

Writing run_experiment.py


In [7]:
%%writefile analyze.py
#!/usr/bin/env python3
"""
S03 analysis: raw runs -> medians, derived metrics, optimal-split table, manual-format CSV, figures.

Reads  results/s03_raw_runs.csv  (written by run_experiment.py)
Writes results/s03_results_median.csv, results/s03_optimal_split.csv,
       results/s03_results_manual_format.csv, results/figures/*.png|svg

All derived metrics (speedup, efficiency, utilisation, ...) are computed here from the raw measurements.

Definitions
  Execution_Time      makespan of one run in ms (start of GPU enqueue -> CPU and GPU both finished)
  Speedup             median CPU-serial makespan (1 thread, CPU share 100%) / makespan
  Speedup_vs_GPU_only median GPU-only makespan (CPU share 0%, same sweep) / makespan
  Resource_Count      CPU threads actually given work + 1 if the GPU got work
  Efficiency          Speedup / Resource_Count (the GPU counts as one resource unit; not comparable with
                      a pure CPU parallel efficiency)
  Partition efficiency ideal makespan / measured makespan, where ideal = 1 / (1/T_cpu_only + 1/T_gpu_only)
                      (perfect overlap of the two devices, no extra overhead)
  CPU utilisation     sum of per-thread busy time / (threads x makespan)
  GPU kernel util.    summed kernel time / makespan;   GPU active = GPU span (H2D+kernel+D2H) / makespan
"""
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Okabe-Ito colour-blind-safe palette
C_CPU, C_GPU, C_HYB, C_RED, C_GREY, C_PURPLE = "#E69F00", "#0072B2", "#009E73", "#D55E00", "#6b6b6b", "#CC79A7"

BASE_KEY = ["workload", "tiles", "tile_dim", "passes", "streams", "chunk_tiles", "pinned"]
CFG_KEY = BASE_KEY + ["threads_req"]
NUMERIC = ["tiles", "tile_dim", "passes", "threads_req", "threads_actual", "streams", "chunk_tiles", "pinned", "cpu_frac",
           "tiles_cpu", "tiles_gpu", "rep", "makespan_ms", "sched_ms", "cpu_wall_ms", "cpu_busy_sum_ms", "gpu_h2d_ms",
           "gpu_kernel_ms", "gpu_d2h_ms", "gpu_span_ms", "sync_wait_ms", "max_rel_err", "host_bytes", "device_bytes"]


def load(path):
    raw = pd.read_csv(path)
    for c in NUMERIC:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    raw["cpu_frac"] = raw["cpu_frac"].round(4)
    return raw


def add_baselines(df):
    """df has one row per (config, cpu_frac) with median metrics. Adds baselines and derived metrics."""
    cpu_only = df[df.cpu_frac == 1.0].set_index(CFG_KEY)["makespan_ms"].rename("cpu_only_ms")
    gpu_only = df[df.cpu_frac == 0.0].set_index(CFG_KEY)["makespan_ms"].rename("gpu_only_ms")
    serial = (df[(df.cpu_frac == 1.0) & (df.threads_req == 1)].set_index(BASE_KEY)["makespan_ms"].rename("serial_ms"))
    df = df.join(cpu_only, on=CFG_KEY).join(gpu_only, on=CFG_KEY).join(serial, on=BASE_KEY)
    df["speedup_vs_serial"] = df.serial_ms / df.makespan_ms
    df["speedup_vs_gpu_only"] = df.gpu_only_ms / df.makespan_ms
    df["speedup_vs_cpu_only"] = df.cpu_only_ms / df.makespan_ms
    df["resource_count"] = np.where(df.tiles_cpu > 0, df.threads_req, 0) + np.where(df.tiles_gpu > 0, 1, 0)
    df["efficiency"] = df.speedup_vs_serial / df.resource_count
    df["ideal_ms"] = 1.0 / (1.0 / df.cpu_only_ms + 1.0 / df.gpu_only_ms)
    df["partition_eff"] = df.ideal_ms / df.makespan_ms
    thr = df.threads_req.where(df.tiles_cpu > 0, np.nan)
    df["cpu_util"] = (df.cpu_busy_sum_ms / (thr * df.makespan_ms)).fillna(0.0)
    df["gpu_kernel_util"] = df.gpu_kernel_ms / df.makespan_ms
    df["gpu_active"] = df.gpu_span_ms / df.makespan_ms
    df["comm_ms"] = df.gpu_h2d_ms + df.gpu_d2h_ms
    df["transfer_frac"] = (df.comm_ms / df.gpu_span_ms).where(df.gpu_span_ms > 0, np.nan)
    df["imbalance_ms"] = (df.cpu_wall_ms - df.gpu_span_ms).abs().where((df.tiles_cpu > 0) & (df.tiles_gpu > 0), np.nan)
    return df


def summarise(raw):
    g = raw.groupby(CFG_KEY + ["cpu_frac"])
    cols = [c for c in NUMERIC if c not in CFG_KEY + ["cpu_frac", "rep"]]
    med = g[cols].median().reset_index()
    spread = g["makespan_ms"].agg(["count", "min", "max", "mean", "std"]).reset_index()
    spread.columns = CFG_KEY + ["cpu_frac", "n_reps", "makespan_min", "makespan_max", "makespan_mean", "makespan_std"]
    q = g["makespan_ms"].quantile([0.25, 0.75]).unstack().reset_index()
    q.columns = CFG_KEY + ["cpu_frac", "makespan_q1", "makespan_q3"]
    med = med.merge(spread, on=CFG_KEY + ["cpu_frac"]).merge(q, on=CFG_KEY + ["cpu_frac"])
    med["makespan_cv"] = med.makespan_std / med.makespan_mean
    return add_baselines(med)


def optimal_table(med):
    rows = []
    for key, g in med.groupby(CFG_KEY):
        g = g.sort_values("cpu_frac")
        if g.cpu_frac.min() > 0 or g.cpu_frac.max() < 1:
            continue
        best = g.loc[g.makespan_ms.idxmin()]
        near = g[g.makespan_ms <= 1.02 * best.makespan_ms]
        gpu = g[g.cpu_frac == 0].iloc[0]
        cpu = g[g.cpu_frac == 1].iloc[0]
        hyb = g[(g.cpu_frac > 0) & (g.cpu_frac < 1)]
        best_h = hyb.loc[hyb.makespan_ms.idxmin()] if len(hyb) else best
        gain = (gpu.makespan_ms - best_h.makespan_ms) / gpu.makespan_ms
        noise = 2 * max(np.nan_to_num(best_h.makespan_cv), np.nan_to_num(gpu.makespan_cv))
        single = gpu if gpu.makespan_ms <= cpu.makespan_ms else cpu
        gain_single = (single.makespan_ms - best_h.makespan_ms) / single.makespan_ms
        noise_single = 2 * max(np.nan_to_num(best_h.makespan_cv), np.nan_to_num(single.makespan_cv))
        pred = gpu.makespan_ms / (cpu.makespan_ms + gpu.makespan_ms)
        rows.append(dict(zip(CFG_KEY, key)) | dict(
            gpu_only_ms=gpu.makespan_ms, cpu_only_ms=cpu.makespan_ms, serial_ms=best.serial_ms,
            best_frac=best.cpu_frac, best_ms=best.makespan_ms,
            best_hybrid_frac=best_h.cpu_frac, best_hybrid_ms=best_h.makespan_ms,
            near_opt_lo=near.cpu_frac.min(), near_opt_hi=near.cpu_frac.max(),
            speedup_best_vs_gpu_only=gpu.makespan_ms / best.makespan_ms,
            speedup_best_vs_cpu_only=cpu.makespan_ms / best.makespan_ms,
            speedup_best_vs_serial=best.serial_ms / best.makespan_ms,
            hybrid_gain_vs_gpu_only=gain, noise_threshold=noise,
            hybrid_beats_gpu_only=bool(gain > max(noise, 0.02)),
            hybrid_gain_vs_best_single=gain_single,
            hybrid_beats_best_single=bool(gain_single > max(noise_single, 0.02)),
            predicted_frac=pred, ideal_ms=best.ideal_ms, partition_eff_best=best.partition_eff,
            faster_single_device="GPU-only" if gpu.makespan_ms <= cpu.makespan_ms else "CPU-only",
            gpu_only_transfer_frac=gpu.transfer_frac, cv_best=best.makespan_cv, cv_gpu_only=gpu.makespan_cv,
            sync_ms_at_best=best.sync_wait_ms, sched_ms_at_best=best.sched_ms,
            cpu_util_at_best=best.cpu_util, gpu_active_at_best=best.gpu_active))
    return pd.DataFrame(rows)


def manual_format(raw, med):
    """One CSV row per run with the columns required by the case-study manual (+ extras)."""
    m = med[CFG_KEY + ["cpu_frac", "serial_ms", "gpu_only_ms", "cpu_only_ms"]]
    r = raw.merge(m, on=CFG_KEY + ["cpu_frac"], how="left")
    used_thr = np.where(r.tiles_cpu > 0, r.threads_req, 0)
    r["Resource_Count"] = used_thr + np.where(r.tiles_gpu > 0, 1, 0)
    r["Speedup"] = r.serial_ms / r.makespan_ms
    model = np.where(r.tiles_gpu == 0, "OpenMP (CPU-only)", np.where(r.tiles_cpu == 0, "CUDA (GPU-only)", "OpenMP+CUDA hybrid"))
    out = pd.DataFrame({
        "Student_ID": r.Student_ID, "Batch_ID": r.Batch_ID,
        "Workload": "tileblur_p" + r.passes.astype(int).astype(str),
        "Dataset": r.Dataset,
        "Problem_Size": (r.tiles * r.tile_dim * r.tile_dim).astype("int64"),
        "Algorithm": "3x3 box blur x" + r.passes.astype(int).astype(str) + " passes, CPU/GPU tile partitioning",
        "Parallel_Model": model, "Hardware": r.Hardware,
        "Resource_Count": r.Resource_Count, "Threads": r.threads_req,
        "Execution_Time": r.makespan_ms,
        "Speedup": r.Speedup, "Efficiency": r.Speedup / r.Resource_Count,
        "Communication_Time": r.gpu_h2d_ms + r.gpu_d2h_ms,
        "Synchronization_Time": r.sync_wait_ms, "Scheduling_Time": r.sched_ms,
        "Memory_Use": (r.host_bytes + r.device_bytes) / 1048576.0,
        "Energy_if_available": np.nan, "Run_ID": r.Run_ID,
        # extras (units: time in ms, Memory_Use in MiB, Problem_Size in pixels)
        "Tiles": r.tiles.astype(int), "Tile_Dim": r.tile_dim.astype(int), "Passes": r.passes.astype(int),
        "CPU_Fraction": r.cpu_frac, "Tiles_CPU": r.tiles_cpu.astype(int), "Tiles_GPU": r.tiles_gpu.astype(int),
        "Streams": r.streams.astype(int), "Pinned_Host_Memory": r.pinned.astype(int), "Repetition": r.rep.astype(int),
        "Baseline_for_Speedup": "CPU serial (1 thread, CPU share 100%), median of repetitions",
        "Speedup_vs_GPU_only": r.gpu_only_ms / r.makespan_ms, "Speedup_vs_CPU_only_same_threads": r.cpu_only_ms / r.makespan_ms,
        "CPU_Compute_Wall_ms": r.cpu_wall_ms, "CPU_Busy_Sum_ms": r.cpu_busy_sum_ms,
        "GPU_H2D_ms": r.gpu_h2d_ms, "GPU_Kernel_ms": r.gpu_kernel_ms, "GPU_D2H_ms": r.gpu_d2h_ms, "GPU_Span_ms": r.gpu_span_ms,
        "Max_Rel_Error": r.max_rel_err})
    return out


# ------------------------------------------------------------------ plotting
def style():
    plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
                         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                         "figure.dpi": 110, "savefig.dpi": 200})


def save(fig, figdir, name):
    fig.savefig(figdir / f"{name}.png", bbox_inches="tight")
    fig.savefig(figdir / f"{name}.svg", bbox_inches="tight")
    plt.close(fig)


def headline(med):
    tiles = med.tiles.max()
    th = med.threads_req.max()
    return med[(med.tiles == tiles) & (med.threads_req == th)].sort_values(["passes", "cpu_frac"]), int(tiles), int(th)


def fig_main(med, figdir):
    h, tiles, th = headline(med)
    pass_list = sorted(h.passes.unique())
    fig, axes = plt.subplots(len(pass_list), 3, figsize=(7.4, 1.95 * len(pass_list) + 0.3), squeeze=False)
    for i, p in enumerate(pass_list):
        g = h[h.passes == p].sort_values("cpu_frac")
        x = g.cpu_frac * 100
        best = g.loc[g.makespan_ms.idxmin()]
        ax = axes[i, 0]
        ax.fill_between(x, g.makespan_min, g.makespan_max, color=C_HYB, alpha=0.18, lw=0)
        ax.plot(x, g.makespan_ms, "o-", color=C_HYB, ms=3, lw=1.5, label="makespan")
        ax.plot(x, g.cpu_wall_ms.where(g.tiles_cpu > 0), "--", color=C_CPU, lw=1, label="CPU part")
        ax.plot(x, g.gpu_span_ms.where(g.tiles_gpu > 0), "--", color=C_GPU, lw=1, label="GPU part (H2D+kernel+D2H)")
        ax.plot([best.cpu_frac * 100], [best.makespan_ms], "*", color=C_RED, ms=9, zorder=5, label="best split")
        ax.annotate(f"best: {best.cpu_frac * 100:.0f}% CPU", (best.cpu_frac * 100, best.makespan_ms), textcoords="offset points",
                    xytext=(0, -13 if best.cpu_frac < 0.5 else 8), ha="center", fontsize=6.5, color=C_RED)
        ax.set_ylabel(f"time (ms)\npasses = {int(p)}")
        ax.set_yscale("log")
        if i == 0:
            ax.set_title("Time vs CPU share")
        if i == 0:
            ax.legend(loc="best", frameon=True, framealpha=0.85, edgecolor="none", fontsize=5.5)
        ax = axes[i, 1]
        ax.plot(x, g.cpu_util * 100, "o-", color=C_CPU, ms=3, label="CPU busy")
        ax.plot(x, g.gpu_kernel_util * 100, "s-", color=C_GPU, ms=3, label="GPU kernel")
        ax.plot(x, g.gpu_active * 100, "^--", color=C_PURPLE, ms=3, lw=1, label="GPU active (incl. copies)")
        ax.axvline(best.cpu_frac * 100, color=C_RED, lw=0.8, ls=":")
        ax.set_ylim(-3, 108)
        if i == 0:
            ax.set_title("Utilisation vs CPU share")
        ax.set_ylabel("% of makespan")
        if i == 0:
            ax.legend(loc="best", frameon=True, framealpha=0.85, edgecolor="none", fontsize=5.5)
        ax = axes[i, 2]
        ax.plot(x, g.speedup_vs_serial, "o-", color=C_HYB, ms=3, label="vs CPU serial")
        ax.plot(x, g.speedup_vs_gpu_only, "s-", color=C_GPU, ms=3, label="vs GPU-only")
        ax.axhline(1, color=C_GREY, lw=0.8)
        ax.axvline(best.cpu_frac * 100, color=C_RED, lw=0.8, ls=":")
        ax.set_yscale("log")
        if i == 0:
            ax.set_title("Speedup vs CPU share")
        ax.set_ylabel("speedup (x)")
        if i == 0:
            ax.legend(loc="best", frameon=True, framealpha=0.85, edgecolor="none", fontsize=5.5)
        if i == len(pass_list) - 1:
            for a in axes[i]:
                a.set_xlabel("share of tiles on CPU (%)")
    fig.suptitle(f"{tiles} tiles of {int(h.tile_dim.iloc[0])}x{int(h.tile_dim.iloc[0])}, {th} CPU threads (median of repetitions, band = min-max)", fontsize=8, y=1.0)
    fig.tight_layout()
    save(fig, figdir, "fig1_split_time_util_speedup")


def fig_breakdown(med, figdir):
    h, tiles, th = headline(med)
    pass_list = sorted(h.passes.unique())
    fig, axes = plt.subplots(1, len(pass_list), figsize=(7.4, 2.35), squeeze=False)
    for j, p in enumerate(pass_list):
        g = h[h.passes == p].sort_values("cpu_frac").reset_index(drop=True)
        ax = axes[0, j]
        x = np.arange(len(g))
        w = 0.38
        ax.bar(x - w / 2, g.cpu_wall_ms, w, color=C_CPU, label="CPU compute")
        ax.bar(x + w / 2, g.gpu_h2d_ms, w, color="#56B4E9", label="GPU H2D copy")
        ax.bar(x + w / 2, g.gpu_kernel_ms, w, bottom=g.gpu_h2d_ms, color=C_GPU, label="GPU kernels")
        ax.bar(x + w / 2, g.gpu_d2h_ms, w, bottom=g.gpu_h2d_ms + g.gpu_kernel_ms, color="#003f66", label="GPU D2H copy")
        ax.plot(x, g.makespan_ms, "k_", ms=9, mew=1.6, label="makespan")
        ax.set_xticks(x)
        ax.set_xticklabels([f"{f * 100:.0f}" for f in g.cpu_frac])
        ax.set_xlabel("share of tiles on CPU (%)")
        ax.set_title(f"passes = {int(p)}")
        if j == 0:
            ax.set_ylabel("time (ms)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.suptitle(f"Where the time goes: CPU compute vs GPU copy and kernel time ({tiles} tiles, {th} threads)", fontsize=8, y=1.02)
    fig.tight_layout()
    fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False, fontsize=6.5, bbox_to_anchor=(0.5, -0.1))
    save(fig, figdir, "fig2_time_breakdown")


def fig_optimal(opt, figdir):
    if opt.empty:
        return
    th_max, t_max = opt.threads_req.max(), opt.tiles.max()
    pass_list = sorted(opt.passes.unique())
    cols = [C_GPU, C_CPU, C_HYB, C_RED, C_PURPLE]
    fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.25))
    ax = axes[0]
    for k, p in enumerate(pass_list):
        g = opt[(opt.passes == p) & (opt.tiles == t_max)].sort_values("threads_req")
        ax.plot(g.threads_req, g.best_frac * 100, "o-", color=cols[k % 5], ms=4, label=f"measured, passes {int(p)}")
        ax.plot(g.threads_req, g.predicted_frac * 100, "x--", color=cols[k % 5], ms=5, lw=0.9, label=f"predicted, passes {int(p)}")
    ax.set_xlabel("CPU threads")
    ax.set_ylabel("optimal CPU share (%)")
    ax.set_title(f"Optimal share vs threads ({int(t_max)} tiles)")
    ax.set_xticks(sorted(opt.threads_req.unique()))
    ax.legend(frameon=False, fontsize=5.5)
    ax = axes[1]
    for k, p in enumerate(pass_list):
        g = opt[(opt.passes == p) & (opt.threads_req == th_max)].sort_values("tiles")
        ax.plot(g.tiles, g.best_frac * 100, "o-", color=cols[k % 5], ms=4, label=f"measured, passes {int(p)}")
        ax.plot(g.tiles, g.predicted_frac * 100, "x--", color=cols[k % 5], ms=5, lw=0.9)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("problem size (tiles)")
    ax.set_title(f"Optimal share vs size ({int(th_max)} threads)")
    ax.legend(frameon=False, fontsize=5.5)
    ax = axes[2]
    for k, p in enumerate(pass_list):
        g = opt[(opt.passes == p) & (opt.tiles == t_max)].sort_values("threads_req")
        ax.plot(g.threads_req, g.speedup_best_vs_gpu_only, "o-", color=cols[k % 5], ms=4, label=f"vs GPU-only, passes {int(p)}")
        ax.plot(g.threads_req, g.speedup_best_vs_cpu_only, "s--", color=cols[k % 5], ms=3.5, lw=0.9, label=f"vs CPU-only, passes {int(p)}")
    ax.axhline(1, color=C_GREY, lw=0.8)
    ax.set_yscale("log")
    ax.set_xlabel("CPU threads")
    ax.set_ylabel("speedup of best split (x)")
    ax.set_title("Gain of best split")
    ax.set_xticks(sorted(opt.threads_req.unique()))
    ax.legend(frameon=False, fontsize=5.5)
    fig.tight_layout()
    save(fig, figdir, "fig3_optimal_split_trends")


def fig_all_curves(med, figdir):
    pass_list = sorted(med.passes.unique())
    tile_list = sorted(med.tiles.unique())
    fig, axes = plt.subplots(len(pass_list), len(tile_list), figsize=(3.0 * len(tile_list), 2.3 * len(pass_list)), squeeze=False)
    cm = plt.get_cmap("viridis")
    ths = sorted(med.threads_req.unique())
    for i, p in enumerate(pass_list):
        for j, n in enumerate(tile_list):
            ax = axes[i, j]
            for k, t in enumerate(ths):
                g = med[(med.passes == p) & (med.tiles == n) & (med.threads_req == t)].sort_values("cpu_frac")
                ax.plot(g.cpu_frac * 100, g.makespan_ms, "o-", ms=2.5, lw=1.2, color=cm(k / max(1, len(ths) - 1) * 0.85), label=f"{int(t)} threads")
            ax.set_yscale("log")
            ax.set_title(f"passes {int(p)}, {int(n)} tiles")
            if j == 0:
                ax.set_ylabel("makespan (ms)")
            if i == len(pass_list) - 1:
                ax.set_xlabel("CPU share (%)")
            if i == 0 and j == 0:
                ax.legend(frameon=False, fontsize=6)
    fig.tight_layout()
    save(fig, figdir, "fig4_all_configurations")


def fig_overhead(med, figdir):
    h, tiles, th = headline(med)
    pass_list = sorted(h.passes.unique())
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.3))
    for k, p in enumerate(pass_list):
        g = h[h.passes == p].sort_values("cpu_frac")
        c = [C_GPU, C_CPU, C_HYB, C_RED][k % 4]
        axes[0].plot(g.cpu_frac * 100, g.sync_wait_ms, "o-", ms=3, color=c, label=f"passes {int(p)}")
        axes[1].plot(g.cpu_frac * 100, g.sched_ms, "o-", ms=3, color=c, label=f"passes {int(p)}")
    axes[0].set_title("Host wait for GPU at the end (synchronisation)")
    axes[1].set_title("Partition + GPU enqueue time (scheduling)")
    for a in axes:
        a.set_xlabel("share of tiles on CPU (%)")
        a.set_ylabel("ms")
        a.legend(frameon=False)
    fig.tight_layout()
    save(fig, figdir, "fig5_sync_and_scheduling")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", default="results/s03_raw_runs.csv")
    ap.add_argument("--outdir", default="results")
    a = ap.parse_args()
    out = Path(a.outdir)
    figdir = out / "figures"
    figdir.mkdir(parents=True, exist_ok=True)
    style()

    raw = load(a.input)
    med = summarise(raw)
    opt = optimal_table(med)
    med.to_csv(out / "s03_results_median.csv", index=False)
    opt.to_csv(out / "s03_optimal_split.csv", index=False)
    manual_format(raw, med).to_csv(out / "s03_results_manual_format.csv", index=False)

    bad = raw[(raw.max_rel_err.notna()) & (raw.max_rel_err > 1e-5)]
    print(f"{len(raw)} runs, {raw.groupby(CFG_KEY).ngroups} configurations, correctness failures: {len(bad)} "
          f"(max error {raw.max_rel_err.max():.2e})")
    fig_main(med, figdir)
    fig_breakdown(med, figdir)
    fig_optimal(opt, figdir)
    fig_all_curves(med, figdir)
    fig_overhead(med, figdir)

    show = opt[["passes", "tiles", "threads_req", "gpu_only_ms", "cpu_only_ms", "best_frac", "best_ms",
                "speedup_best_vs_gpu_only", "predicted_frac", "hybrid_beats_gpu_only"]].round(3)
    print(show.to_string(index=False))
    print(f"\nFiles written to {out}/ and {figdir}/")


if __name__ == "__main__":
    main()

Writing analyze.py


In [8]:
%%writefile make_report.py
#!/usr/bin/env python3
"""
Builds the individual technical report (Word .docx) for CS7 / S03 from the experiment output.

All numbers in the text, the summary table and the figures come from results/ (written by
run_experiment.py + analyze.py). Static sections (introduction, literature, method, limitations,
future work) are fixed text. Sentences that interpret the data are built from the measured numbers
and branch on what was actually found, so READ THE RESULT PARAGRAPHS BEFORE SUBMITTING and edit
anything that does not match what you saw in the figures.

Usage:  python make_report.py --name "Your Name" --reg 23BCE0434 --batch "B07"
"""
import argparse, re
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from docx import Document
    from docx.shared import Pt, Inches, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
except ImportError:
    raise SystemExit("python-docx is needed:  pip install python-docx")

FONT = "Times New Roman"

REFS = [
    'NVIDIA, "CUDA C++ Best Practices Guide," CUDA Toolkit Documentation, v13.0. https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html',
    'S. Mittal and J. S. Vetter, "A Survey of CPU-GPU Heterogeneous Computing Techniques," ACM Computing Surveys, vol. 47, no. 4, Art. 69, 2015. https://doi.org/10.1145/2788396',
    'C.-K. Luk, S. Hong, and H. Kim, "Qilin: Exploiting Parallelism on Heterogeneous Multiprocessors with Adaptive Mapping," in Proc. 42nd IEEE/ACM Int. Symp. on Microarchitecture (MICRO-42), 2009, pp. 45-55.',
    'C. Gregg and K. Hazelwood, "Where is the Data? Why You Cannot Debate CPU vs. GPU Performance Without the Answer," in Proc. IEEE ISPASS, 2011. https://doi.org/10.1109/ISPASS.2011.5762730',
    'R. Nozal and J. L. Bosque, "Exploiting co-execution with oneAPI: heterogeneity from a modern perspective," arXiv:2106.01726, 2021 (Springer version: https://doi.org/10.1007/978-3-030-85665-6_31).',
    'Z. Li, M. Paolieri, and L. Golubchik, "Accelerating Mobile Inference through Fine-Grained CPU-GPU Co-Execution," in Selected Papers of EPEW 2025, LNCS vol. 15657, Springer, 2026, pp. 41-55.',
    'F. Wang et al., "Accurate and Reliable Energy Measurement and Modelling of Data Transfer Between CPU and GPU in Parallel Applications on Heterogeneous Hybrid Platforms," IEEE Trans. Computers, vol. 74, no. 3, pp. 1011-1024, 2025. https://doi.org/10.1109/TC.2024.3504262',
]


# ------------------------------------------------------------------ small formatting helpers
def ms(x):
    if x >= 100:
        return f"{x:.0f} ms"
    if x >= 10:
        return f"{x:.1f} ms"
    return f"{x:.2f} ms"


def pc(x, d=0):
    return f"{100 * x:.{d}f}%"


def sx(x):
    return f"{x:.2f}x" if x < 10 else f"{x:.1f}x"


def join_and(items):
    items = list(items)
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


def nums(vals):
    return join_and(str(int(v)) for v in vals)


def shade(cell, hex_fill):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), hex_fill)
    tcPr.append(shd)


def set_font(run, size=None, bold=None, italic=None):
    run.font.name = FONT
    run._element.rPr.rFonts.set(qn("w:eastAsia"), FONT)
    if size:
        run.font.size = Pt(size)
    if bold is not None:
        run.bold = bold
    if italic is not None:
        run.italic = italic


def add_runs(par, text, size=9.5):
    """Supports **bold** segments."""
    for i, chunk in enumerate(re.split(r"\*\*", text)):
        if chunk:
            r = par.add_run(chunk)
            set_font(r, size, bold=(i % 2 == 1))


def para(doc, text, size=9.5, after=3, align=None, italic=False, before=0):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.space_after, pf.space_before, pf.line_spacing = Pt(after), Pt(before), 1.0
    if align == "j":
        p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    elif align == "c":
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    add_runs(p, text, size)
    if italic:
        for r in p.runs:
            r.italic = True
    return p


def heading(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_before, p.paragraph_format.space_after, p.paragraph_format.keep_with_next = Pt(5), Pt(2), True
    r = p.add_run(text)
    set_font(r, 10.5, bold=True)
    r.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F)
    # real outline level so the Word navigation pane works
    pPr = p._p.get_or_add_pPr()
    lvl = OxmlElement("w:outlineLvl")
    lvl.set(qn("w:val"), "0")
    pPr.append(lvl)
    return p


def caption(doc, text):
    p = para(doc, text, size=8, after=4, italic=True, align="c")
    return p


def table(doc, header, rows, widths=None, size=7.5):
    t = doc.add_table(rows=1, cols=len(header))
    t.style = "Table Grid"
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, h in enumerate(header):
        c = t.rows[0].cells[i]
        c.text = ""
        r = c.paragraphs[0].add_run(h)
        set_font(r, size, bold=True)
        shade(c, "D9E2F3")
    for row in rows:
        cells = t.add_row().cells
        for i, v in enumerate(row):
            cells[i].text = ""
            r = cells[i].paragraphs[0].add_run(str(v))
            set_font(r, size)
    for row in t.rows:
        for i, c in enumerate(row.cells):
            for p in c.paragraphs:
                p.paragraph_format.space_after = Pt(0)
                p.paragraph_format.space_before = Pt(0)
            if widths:
                c.width = Inches(widths[i])
    for row in t.rows[:-1]:
        for c in row.cells:
            for p in c.paragraphs:
                p.paragraph_format.keep_with_next = True
    return t


# ------------------------------------------------------------------ facts from the data
def read_sysinfo(path):
    d = {}
    if Path(path).exists():
        for line in open(path):
            if ":" in line:
                k, v = line.split(":", 1)
                d[k.strip()] = v.strip()
    return d


def verdict(ok, partly=False):
    return "supported" if ok else ("partly supported" if partly else "not supported")


def compute_facts(raw, med, opt):
    f = {}
    f["n_cfg"] = len(opt)
    f["tiles_list"] = sorted(int(v) for v in opt.tiles.unique())
    f["passes_list"] = sorted(int(v) for v in opt.passes.unique())
    f["threads_list"] = sorted(int(v) for v in opt.threads_req.unique())
    f["tile_dim"] = int(raw.tile_dim.iloc[0])
    f["reps"] = int(med.n_reps.max())
    f["t_max"], f["th_max"], f["th_min"] = max(f["tiles_list"]), max(f["threads_list"]), min(f["threads_list"])
    f["p_min"], f["p_max"] = min(f["passes_list"]), max(f["passes_list"])
    opt = opt.copy()
    opt["opt_mid"] = (opt.near_opt_lo + opt.near_opt_hi) / 2
    f["opt"] = opt
    H = opt[(opt.tiles == f["t_max"]) & (opt.threads_req == f["th_max"])].sort_values("passes")
    f["H"] = H
    f["interior"] = int(((opt.best_frac > 0) & (opt.best_frac < 1)).sum())
    f["beats_gpu"] = int(opt.hybrid_beats_gpu_only.sum())
    f["beats_single"] = int(opt.hybrid_beats_best_single.sum())
    f["gpu_faster"] = int((opt.faster_single_device == "GPU-only").sum())
    f["gain_min"], f["gain_max"] = opt.speedup_best_vs_gpu_only.min(), opt.speedup_best_vs_gpu_only.max()

    def mean_delta(sel_cols, hi, lo, key):
        ds = []
        for _, g in opt.groupby(sel_cols):
            a, b = g[g[key] == hi], g[g[key] == lo]
            if len(a) and len(b):
                ds.append(a.opt_mid.iloc[0] - b.opt_mid.iloc[0])
        return float(np.mean(ds)) if ds else float("nan")

    f["d_passes"] = mean_delta(["tiles", "threads_req"], f["p_max"], f["p_min"], "passes") if f["p_max"] != f["p_min"] else float("nan")
    f["d_threads"] = mean_delta(["tiles", "passes"], f["th_max"], f["th_min"], "threads_req") if f["th_max"] != f["th_min"] else float("nan")
    f["d_size"] = mean_delta(["passes", "threads_req"], f["t_max"], min(f["tiles_list"]), "tiles") if f["t_max"] != min(f["tiles_list"]) else float("nan")
    f["pred_in_range"] = int(((opt.predicted_frac >= opt.near_opt_lo - 0.05) & (opt.predicted_frac <= opt.near_opt_hi + 0.05)).sum())
    f["pred_mae"] = float((opt.predicted_frac - opt.best_frac).abs().mean())
    f["pe_median"] = float(opt.partition_eff_best.median())
    f["sync_rel"] = float((opt.sync_ms_at_best / opt.best_ms).median())
    f["sched_rel"] = float((opt.sched_ms_at_best / opt.best_ms).median())
    f["cv_med"], f["cv_max"] = float(med.makespan_cv.median()), float(med.makespan_cv.max())
    f["err_max"] = float(raw.max_rel_err.max())
    f["n_runs"] = len(raw)
    fr = np.sort(med.cpu_frac.unique())
    f["step"] = int(round(100 * float(np.median(np.diff(fr))))) if len(fr) > 1 else 10
    f["chunk"] = nums(sorted(raw.chunk_tiles.unique()))
    f["streams_txt"] = nums(sorted(raw.streams.unique()))
    f["streams_phrase"] = f["streams_txt"] + (" stream" if f["streams_txt"] == "1" else " streams")
    pin = set(raw.pinned.unique())
    f["pinned_txt"] = "pinned" if pin == {1} else ("pageable" if pin == {0} else "pinned and pageable")
    # imbalance at the best split of interior-optimum configs
    imb = []
    for _, r in opt.iterrows():
        if 0 < r.best_frac < 1:
            row = med[(med.tiles == r.tiles) & (med.passes == r.passes) & (med.threads_req == r.threads_req) & (med.cpu_frac == r.best_frac)]
            if len(row):
                imb.append(float(row.imbalance_ms.iloc[0] / row.makespan_ms.iloc[0]))
    f["imb_med"] = float(np.median(imb)) if imb else float("nan")
    f["single_best"] = opt[(opt.best_frac == 0) | (opt.best_frac == 1)]
    return f


def trend_word(d, unit="points"):
    if np.isnan(d):
        return "could not be assessed"
    if d <= -0.05:
        return f"fell by about {abs(100 * d):.0f} {unit}"
    if d >= 0.05:
        return f"rose by about {100 * d:.0f} {unit}"
    return "stayed roughly constant"


def build_texts(f, info):
    H = f["H"]
    t = {}
    per_pass = []
    for _, r in H.iterrows():
        per_pass.append(f"Passes {int(r.passes)}: best share {pc(r.best_frac)} CPU, {ms(r.best_ms)} versus GPU-only {ms(r.gpu_only_ms)} ({sx(r.speedup_best_vs_gpu_only)}) and CPU-only {ms(r.cpu_only_ms)} ({sx(r.speedup_best_vs_cpu_only)})")
    t["per_pass"] = per_pass

    h1_interior = f["interior"] >= 0.5 * f["n_cfg"]
    h1_trend = (not np.isnan(f["d_passes"])) and f["d_passes"] <= -0.05
    h2_rate = f["pred_in_range"] / f["n_cfg"]
    t["H1v"] = verdict(h1_interior and h1_trend, h1_interior or h1_trend)
    t["H2v"] = verdict(h2_rate >= 0.7, h2_rate >= 0.4)
    not_always = (f["beats_gpu"] < f["n_cfg"]) or (f["beats_single"] < f["n_cfg"])
    t["H3v"] = verdict(not_always)

    t["abstract"] = (
        f"A discrete GPU leaves the multicore CPU idle, but splitting a job across both devices adds copy and synchronisation costs. "
        f"This study measures how the share of work given to the CPU changes the makespan of an independent-task workload and where the optimum lies. "
        f"The workload is {nums(f['tiles_list'])} tiles of {f['tile_dim']}x{f['tile_dim']} floats, each filtered by {nums(f['passes_list'])} passes of a 3x3 blur. "
        f"A share of tiles from 0 to 100 percent runs on {nums(f['threads_list'])} OpenMP threads and the rest on {info.get('gpu_short', 'the GPU')} through CUDA. "
        f"Medians of {f['reps']} repetitions are compared with CPU-serial, CPU-only and GPU-only baselines. "
        f"At the largest size with {f['th_max']} threads the best split sent {join_and(pc(v) for v in H.best_frac)} of the tiles to the CPU "
        f"(passes {nums(H.passes)} respectively), a speedup of {join_and(sx(v) for v in H.speedup_best_vs_gpu_only)} over GPU-only. "
        f"Over all {f['n_cfg']} configurations the hybrid beat the faster single device by more than noise in {f['beats_single']}. "
        f"The optimal share {trend_word(f['d_passes'])} as passes rose from {f['p_min']} to {f['p_max']}, "
        f"and the rule f* = T_gpu / (T_cpu + T_gpu) landed in the near-optimal range in {f['pred_in_range']} of {f['n_cfg']} configurations."
    )
    return t


# ------------------------------------------------------------------ document
def build(args):
    res = Path(args.results)
    raw = pd.read_csv(res / "s03_raw_runs.csv")
    med = pd.read_csv(res / "s03_results_median.csv")
    opt = pd.read_csv(res / "s03_optimal_split.csv")
    sysinfo = read_sysinfo(res / "system_info.txt")
    figdir = res / "figures"
    hardware = str(raw.Hardware.iloc[0])
    if "NO GPU" in hardware and not args.allow_test_data:
        raise SystemExit("These results come from the host-only test build (no GPU). Run the real experiment on Kaggle with a GPU first.")
    f = compute_facts(raw, med, opt)
    gpu_line = sysinfo.get("GPU", "")
    gpu_name = gpu_line.split("|")[0].strip() if gpu_line else hardware.split("+")[0].strip()
    info = {"gpu_short": f"an {gpu_name}" if gpu_name[:1].upper() in "AEIOU" else f"a {gpu_name}"}
    t = build_texts(f, info)
    H, opt = f["H"], f["opt"]

    doc = Document()
    sec = doc.sections[0]
    sec.page_width, sec.page_height = Inches(8.27), Inches(11.69)
    sec.left_margin = sec.right_margin = Inches(0.7)
    sec.top_margin, sec.bottom_margin = Inches(0.6), Inches(0.6)
    st = doc.styles["Normal"]
    st.font.name = FONT
    st.element.rPr.rFonts.set(qn("w:eastAsia"), FONT)
    st.font.size = Pt(9.5)

    # title block
    p = para(doc, "**CS7 - S03: CPU-GPU Workload Partitioning**", size=14, after=0, align="c")
    para(doc, "BCSE412L Parallel Computing, Fall Semester 2026-27 | Individual Technical Report", size=9.5, after=1, align="c")
    para(doc, f"**Author:** {args.name}    **Registration No.:** {args.reg}    **Batch / Section:** {args.batch}    **Date:** {args.date}", size=9, after=3, align="c")

    heading(doc, "Abstract")
    para(doc, t["abstract"], align="j")

    heading(doc, "1. Introduction and Literature Survey")
    para(doc, (
        "A discrete GPU can run a data-parallel kernel far faster than a multicore CPU, but the data must cross the PCIe bus and the CPU sits idle while it waits. "
        "The CUDA Best Practices Guide therefore advises minimising host-device transfers and overlapping the remaining ones with computation [1]. "
        "Gregg and Hazelwood show that CPU-versus-GPU comparisons depend strongly on where the data resides and whether transfer time is counted [4], and Mittal and Vetter survey workload partitioning as one of the main techniques for using both processors [2]. "
        "Qilin automates the CPU/GPU mapping with an empirically trained performance model [3]. Nozal and Bosque extend oneAPI so that a single kernel co-executes on a CPU and an integrated GPU, and report that dynamic load balancing improves efficiency over static splits [5]. "
        "Li et al. split each neural-network layer by output channels between CPU and GPU and use latency predictors to choose the split that balances the two devices [6]. "
        "Recent work also models the cost, including energy, of CPU-GPU data transfer [7]."), align="j")
    para(doc, (
        "**Gap addressed.** These works study runtimes, predictors or one application. This study instead isolates one question on a discrete PCIe GPU: for independent tiles, how does the best static CPU share move with arithmetic intensity, "
        "CPU thread count and problem size, and is a rule built from two single-device timings enough to find it? It extends the CPU-versus-GPU crossover measured in S02 of this batch by examining the region between the two extremes."), align="j")

    heading(doc, "2. Problem, Research Question and Hypotheses")
    para(doc, (
        "**Problem.** A workload of N independent image tiles must be processed with the smallest makespan, using the CPU, the GPU, or both. "
        "**Research question (CS7 S03).** Is CPU+GPU execution always better than GPU-only execution, and what split minimises the makespan? "
        "**H1:** the optimal CPU share is neither 0 nor 100 percent and falls as arithmetic intensity (passes per tile) rises, because kernel work grows with passes while copy cost per tile does not. "
        "**H2:** the throughput-proportional share f* = T_gpu / (T_cpu + T_gpu), computed from the two single-device runs, predicts the measured optimum. "
        "**H3:** hybrid execution does not always beat GPU-only; where the CPU is slow relative to the GPU the gain disappears into noise."), align="j")

    heading(doc, "3. Methodology")
    cpu_line = sysinfo.get("CPU", "").split("|")[0].strip() or "CPU not recorded"
    nv = re.search(r"release ([\d.]+), (V[\d.]+)", sysinfo.get("CUDA compiler", ""))
    cuda_txt = f"nvcc {nv.group(1)} ({nv.group(2)})" if nv else "CUDA compiler version not recorded"
    clean = lambda s: s.replace(" | ", ", ")
    para(doc, (
        f"**Workload.** {nums(f['tiles_list'])} tiles of {f['tile_dim']}x{f['tile_dim']} single-precision pixels ({nums([v * f['tile_dim'] ** 2 * 4 / 2 ** 20 for v in f['tiles_list']])} MiB of input), "
        f"each processed by {nums(f['passes_list'])} passes of a 3x3 box blur with clamped edges inside the tile. Passes set the arithmetic intensity while the data volume stays fixed. Tiles are independent, so any subset can run on either device. "
        f"**Partition.** A share f of the tiles goes to the CPU (OpenMP, {nums(f['threads_list'])} threads, dynamic scheduling) and 1-f to the GPU (CUDA, {f['chunk']}-tile chunks: host-to-device copy, one kernel per pass, device-to-host copy, {f['pinned_txt']} host memory, {f['streams_phrase']}). "
        "The host enqueues the GPU work asynchronously, computes its own share, then waits for the GPU. f = 0 is the GPU-only baseline and f = 1 the CPU-only baseline; CPU-serial (1 thread, f = 1) is the reference for speedup. "
        f"CPU shares from 0 to 100 percent were swept in {f['step']}-point steps. "
        f"**Measurements.** Per run: makespan, enqueue (scheduling) time, CPU compute time and summed per-thread busy time, GPU copy and kernel times from CUDA events, and the host wait for the GPU (synchronisation). "
        f"Each point was repeated {f['reps']} times, alternating sweep direction; the median is reported and the spread is shown as min-max. GPU output was checked against the CPU result for every share (maximum relative error {f['err_max']:.1e}). "
        "Metrics: speedup = baseline median / makespan (baseline always named); partition efficiency = ideal makespan 1/(1/T_cpu + 1/T_gpu) divided by measured makespan; CPU utilisation = busy thread time / (threads x makespan); GPU kernel utilisation = kernel time / makespan. "
        f"**Environment.** GPU: {clean(sysinfo.get('GPU', gpu_name))}; CPU: {clean(sysinfo.get('CPU', cpu_line))}; RAM {sysinfo.get('RAM', 'n/a')}; {cuda_txt}; {sysinfo.get('Host compiler', 'n/a')}. "
        "Build: nvcc -O3 with host flags -fopenmp -O3 -march=native. Source code, raw CSV and plotting scripts are in the submitted archive; its README lists the exact commands."), align="j")

    heading(doc, "4. Results and Discussion")
    rows = []
    for _, r in opt[(opt.threads_req == f["th_max"])].sort_values(["passes", "tiles"]).iterrows():
        rows.append([int(r.passes), int(r.tiles), f"{r.cpu_only_ms:.1f}", f"{r.gpu_only_ms:.1f}",
                     (pc(r.best_frac) if r.near_opt_lo == r.near_opt_hi else f"{pc(r.best_frac)} ({100 * r.near_opt_lo:.0f}-{100 * r.near_opt_hi:.0f}%)"), f"{r.best_ms:.1f}",
                     sx(r.speedup_best_vs_gpu_only), sx(r.speedup_best_vs_cpu_only), sx(r.speedup_best_vs_serial) if pd.notna(r.speedup_best_vs_serial) else "n/a",
                     pc(r.predicted_frac)])
    para(doc, f"**Table 1.** Summary at {f['th_max']} CPU threads (medians, times in ms). Best share = CPU share with the lowest median makespan; range = shares within 2% of it. Speedups name their baseline in the column header.", size=8, after=2, italic=False)
    table(doc, ["Passes", "Tiles", "CPU-only", "GPU-only", "Best CPU share (range)", "Best", "vs GPU-only", "vs CPU-only", "vs CPU serial", "Predicted share f*"], rows,
          widths=[0.5, 0.5, 0.65, 0.65, 1.2, 0.55, 0.7, 0.7, 0.7, 0.75])
    para(doc, "", size=3, after=2)

    doc.add_picture(str(figdir / "fig1_split_time_util_speedup.png"), width=Inches(6.7))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    caption(doc, "Figure 1. Makespan (with the CPU and GPU parts), utilisation and speedup against the share of tiles given to the CPU, at the largest size and thread count. Star = best share.")

    # headline paragraph
    s1 = "; ".join(t["per_pass"])
    para(doc, (
        f"**Headline result** ({f['t_max']} tiles, {f['th_max']} threads). {s1}. "
        f"Across all {f['n_cfg']} configurations the best hybrid split beat GPU-only by more than the noise threshold (twice the larger coefficient of variation, at least 2%) in {f['beats_gpu']} and beat the faster of the two single devices in {f['beats_single']}; "
        f"the speedup of the best split over GPU-only ranged from {sx(f['gain_min'])} to {sx(f['gain_max'])}. The GPU was the faster single device in {f['gpu_faster']} of {f['n_cfg']} configurations."), align="j")

    doc.add_picture(str(figdir / "fig2_time_breakdown.png"), width=Inches(6.7))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    caption(doc, "Figure 2. Time decomposition at each CPU share. Orange: CPU compute wall time. Blue stack: GPU host-to-device copy, kernels and device-to-host copy. Black dash: makespan.")

    tf = H.set_index("passes").gpu_only_transfer_frac
    tf_lo, tf_hi = tf.loc[f["p_min"]], tf.loc[f["p_max"]]
    para(doc, (
        f"**Why the split behaves this way.** In the GPU-only runs, copies took {pc(tf_lo)} of the GPU time at passes {f['p_min']} and {pc(tf_hi)} at passes {f['p_max']}. Copy cost per tile is fixed while kernel cost grows with passes (Figure 2), "
        f"so the GPU becomes cheaper per tile as intensity rises. Measured over all configurations, the optimal CPU share {trend_word(f['d_passes'])} from passes {f['p_min']} to {f['p_max']}, "
        f"{trend_word(f['d_threads'])} from {f['th_min']} to {f['th_max']} threads, and {trend_word(f['d_size'])} from {min(f['tiles_list'])} to {f['t_max']} tiles. "
        + (f"At interior optima the CPU and GPU parts finished within {pc(f['imb_med'])} of the makespan of each other (median), " if not np.isnan(f["imb_med"]) else "")
        + f"and the best split reached a median {pc(f['pe_median'])} of the ideal perfect-overlap makespan. At the best split the host's final wait for the GPU was {pc(f['sync_rel'], 1)} and the enqueue (scheduling) time {pc(f['sched_rel'], 1)} of the makespan (medians)."), align="j")

    doc.add_picture(str(figdir / "fig3_optimal_split_trends.png"), width=Inches(6.7))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    caption(doc, "Figure 3. Optimal CPU share (solid, measured; dashed x, predicted f*) against thread count and problem size, and the gain of the best split over each single device.")

    sb = f["single_best"]
    odd = ""
    if len(sb):
        ex = [f"passes {int(r.passes)}, {int(r.tiles)} tiles, {int(r.threads_req)} thread(s) ({'CPU-only' if r.best_frac == 1 else 'GPU-only'})" for _, r in sb.head(4).iterrows()]
        odd = f" **Unexpected or boundary results.** In {len(sb)} of {f['n_cfg']} configurations a single device was best, for example {join_and(ex)}; these are kept in the data and discussed as cases where the other device adds copy or scheduling cost without removing enough work."
    para(doc, (
        f"**Hypotheses.** H1 is {t['H1v']}: the optimum was interior in {f['interior']} of {f['n_cfg']} configurations and the optimal share {trend_word(f['d_passes'])} from passes {f['p_min']} to {f['p_max']}. "
        f"H2 is {t['H2v']}: f* fell inside the near-optimal range (within 2% of the best makespan, plus or minus 5 points) in {f['pred_in_range']} of {f['n_cfg']} configurations, with a mean absolute difference of {100 * f['pred_mae']:.0f} points from the grid optimum. "
        f"H3 is {t['H3v']}: the hybrid did not beat GPU-only by more than noise in {f['n_cfg'] - f['beats_gpu']} of {f['n_cfg']} configurations and did not beat the faster single device in {f['n_cfg'] - f['beats_single']}. "
        f"Run-to-run variation was a median {pc(f['cv_med'], 1)} (maximum {pc(f['cv_max'], 1)}) coefficient of variation of the makespan.{odd}"), align="j")

    heading(doc, "5. Limitations and Threats to Validity")
    para(doc, (
        f"Only one GPU model, one CPU configuration and a shared cloud notebook were used, so the optimal shares are specific to this hardware (run-to-run variation up to {pc(f['cv_max'], 1)}). "
        "The workload is one regular, memory-light kernel; irregular kernels would shift the balance. The CPU kernel is a hand-vectorised OpenMP loop, not a tuned library, and the GPU kernel is a simple global-memory blur, so the CPU/GPU throughput ratio is implementation-specific. "
        f"Device buffers are allocated once and excluded from the timing; only the host-memory type and stream count listed in Section 3 were tested. Shares were swept on a {f['step']}-point grid, so optima are resolved to about {f['step']} points; the near-optimal range is reported for this reason. "
        "Thread counts above the physical core count share cores with the host thread that drives the GPU."), align="j")

    heading(doc, "6. Conclusion and Future Work")
    if f["beats_single"] == f["n_cfg"]:
        lead = f"For this tile-parallel workload CPU+GPU execution beat the faster single device by more than noise in every one of the {f['n_cfg']} configurations."
    elif f["beats_single"] == 0:
        lead = f"For this tile-parallel workload CPU+GPU execution never beat the faster single device by more than noise in the {f['n_cfg']} configurations."
    else:
        lead = f"For this tile-parallel workload CPU+GPU execution was not uniformly better: it beat the faster single device by more than noise in {f['beats_single']} of {f['n_cfg']} configurations."
    rate = f["pred_in_range"] / f["n_cfg"]
    rule = ("is a usable starting point" if rate >= 0.7 else ("is only a rough guide" if rate >= 0.4 else "did not locate the optimum reliably"))
    concl = (
        f"{lead} At {f['t_max']} tiles and {f['th_max']} threads the best split gave {join_and(sx(v) for v in H.speedup_best_vs_gpu_only)} over GPU-only (passes {nums(H.passes)}). "
        f"The throughput-proportional rule {rule} ({f['pred_in_range']} of {f['n_cfg']} configurations within the near-optimal range). "
        "Future work: (1) replace the static split with dynamic chunk scheduling or work stealing and compare it with the best static split; "
        "(2) overlap copies and kernels with several CUDA streams and measure how far the optimum moves; "
        "(3) calibrate f* online from the first few tiles, as adaptive-mapping systems do [3, 6]; "
        "(4) add energy to the objective, since transfers and idle devices change the energy-optimal split [7].")
    para(doc, concl, align="j")

    heading(doc, "References")
    for i, r in enumerate(REFS, 1):
        para(doc, f"[{i}] {r}", size=7.5, after=0.5)

    doc.save(args.out)
    print(f"saved {args.out}")
    wc = len(t["abstract"].split())
    print(f"abstract length: {wc} words (manual asks for 150-200)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--out", default="S03_Individual_Technical_Report.docx")
    ap.add_argument("--name", default="Mitisha Sharma")
    ap.add_argument("--reg", default="23BCE0434")
    ap.add_argument("--batch", default="________")
    ap.add_argument("--date", default="19 October 2026")
    ap.add_argument("--allow-test-data", action="store_true")
    build(ap.parse_args())

Writing make_report.py


In [9]:
%%writefile README.md
# CS7 S03: CPU-GPU Workload Partitioning (BCSE412L Parallel Computing)

Question: is CPU+GPU execution always better than GPU-only execution, and which split of the work minimises the makespan?

Workload: N independent image tiles (T x T float32). Each tile gets P passes of a 3x3 box blur (clamped edges inside the tile).
A share `f` of the tiles runs on the CPU (OpenMP), the remaining tiles run on the GPU (CUDA, chunked copies, one kernel per pass).
`f = 0` is the GPU-only baseline, `f = 1` is the CPU-only baseline, CPU-serial (1 thread, `f = 1`) is the baseline for speedup.

## Files

| File | Purpose |
|---|---|
| `s03_bench.cu` | CUDA + OpenMP benchmark. Prints one raw CSV row per (CPU share, repetition). |
| `run_experiment.py` | Compiles the benchmark, records the environment, runs the sweep, writes `results/s03_raw_runs.csv`. |
| `analyze.py` | Computes medians and derived metrics from the raw rows, finds the optimal split, writes the CSVs and figures. |
| `make_report.py` | Builds the individual technical report (.docx) from the results. |
| `s03_kaggle.ipynb` | Notebook that writes all of the above and runs everything on Kaggle. |

## Reproduce (Kaggle, GPU accelerator on, Internet on)

```
nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native s03_bench.cu -o s03_bench   # done by run_experiment.py
python run_experiment.py --quick --outdir results_quick     # 2-4 minute sanity run, not for reporting
python run_experiment.py --student-id <REG_NO> --batch-id <BATCH_ID>
python analyze.py
python make_report.py --name "<NAME>" --reg <REG_NO> --batch "<BATCH>"
```

Defaults of the full run: tiles 256, 1024, 4096 (64 MiB, 256 MiB, 1 GiB of input), tile 256 x 256, passes 1 and 16,
CPU threads 1, 2, 4, CPU share 0 to 100% in 10% steps, 7 repetitions, 64-tile chunks, 1 CUDA stream, pinned host memory.
Useful options: `--fine` (5% steps), `--passes 1 4 16`, `--streams 1 4`, `--pinned 0 1`, `--reps 10`.
`run_experiment.py` detects the GPU architecture with `nvidia-smi` (sm_75 for the Tesla T4, sm_60 for a P100).
The host-only build (`--cpu-only-build`) exists to test the scripts without a GPU. Its timings are meaningless and `make_report.py` refuses to use them.

## Outputs (in `results/`)

* `system_info.txt`: GPU, driver, CPU, RAM, compiler versions, compile command, sweep settings.
* `s03_raw_runs.csv`: raw measurements, one row per run (nothing derived).
* `s03_results_manual_format.csv`: one row per run with the columns required by the manual
  (Student_ID, Batch_ID, Workload, Dataset, Problem_Size, Algorithm, Parallel_Model, Hardware, Resource_Count, Threads, Execution_Time,
  Speedup, Efficiency, Communication_Time, Synchronization_Time, Scheduling_Time, Memory_Use, Energy_if_available, Run_ID) plus extra columns.
* `s03_results_median.csv`: medians per point with min, max, quartiles, coefficient of variation, baselines and derived metrics.
* `s03_optimal_split.csv`: one row per configuration: best share, near-optimal range, gains, predicted share, flags for "beats GPU-only" and "beats the faster single device".
* `figures/*.png` and `figures/*.svg`: figures (SVG is the editable form; `analyze.py` is the plotting source).

Units: times in ms, `Memory_Use` in MiB (host input+output plus GPU chunk buffers), `Problem_Size` in pixels (tiles x T x T), `Energy_if_available` is empty (not measured).

## Metric definitions

* `Execution_Time`: makespan of one run, from the start of the GPU enqueue until both CPU and GPU parts have finished.
* `Speedup`: median CPU-serial makespan (1 thread, CPU share 100%) divided by the run's makespan. Baseline is also stated in `Baseline_for_Speedup`.
  `Speedup_vs_GPU_only` and `Speedup_vs_CPU_only_same_threads` give the other two baselines.
* `Resource_Count`: CPU threads that received work, plus 1 if the GPU received work. `Efficiency` = `Speedup` / `Resource_Count`.
  The GPU counts as one resource unit, so this is not comparable with a pure CPU parallel efficiency.
* `Communication_Time`: GPU host-to-device plus device-to-host copy time (CUDA events, summed over chunks). CPU shares use shared memory, so no communication.
* `Synchronization_Time`: time the host spends blocked waiting for the GPU after its own CPU share has finished.
* `Scheduling_Time`: partitioning plus time to enqueue all GPU work.
* Partition efficiency: `1 / (1/T_cpu_only + 1/T_gpu_only)` divided by the measured makespan (perfect overlap, no overhead = 1).
* CPU utilisation: summed per-thread busy time / (threads x makespan). GPU kernel utilisation: summed kernel time / makespan.
* Predicted share: `f* = T_gpu_only / (T_cpu_only + T_gpu_only)`, computed from the two single-device medians.

## Notes

* GPU output is compared with the CPU result for every CPU share (`max_rel_err` in the raw file, first repetition of each point). Runs above 1e-5 are flagged by the driver.
* GPU buffers are allocated once per configuration and excluded from the timing. Host buffers are pinned by default so that copies are asynchronous.
* Repetitions alternate the sweep direction to spread slow drift (clock speed, shared cloud load) over all shares.
* Do not mix results from different hardware without labelling them. `Hardware` is stored in every row.

Writing README.md


In [14]:
!python run_experiment.py --quick --outdir results_quick --student-id {STUDENT_ID} --batch-id {BATCH_ID}
!python analyze.py --input results_quick/s03_raw_runs.csv --outdir results_quick

# import subprocess, time, os
# env = dict(os.environ, OMP_DYNAMIC="FALSE", CUDA_VISIBLE_DEVICES="0")
# fr = "0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1"
# for passes, tiles, th in [(1,64,1),(1,64,4),(8,64,1),(8,64,4),(1,256,1),(1,256,4),(8,256,1),(8,256,4)]:
#     cmd = f"./s03_bench --tiles {tiles} --tile 256 --passes {passes} --threads {th} --reps 1 --fracs {fr} --header 0"
#     s = time.time()
#     try:
#         p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=90, env=env)
#         print(f"passes={passes} tiles={tiles} threads={th}: exit {p.returncode}, {time.time()-s:.1f}s, {len(p.stdout.splitlines())} rows")
#         if p.returncode: print(p.stderr[-500:])
#     except subprocess.TimeoutExpired:
#         print(f"passes={passes} tiles={tiles} threads={th}: TIMED OUT (>90s)")
# print("cpus:", os.cpu_count(), "affinity:", len(os.sched_getaffinity(0)))

+ nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native /kaggle/working/s03_bench.cu -o /kaggle/working/s03_bench
8 configurations x 11 CPU shares x 3 repetitions
[1/8] p1_n64_t1_s1_pin1: 33 rows in 1s, elapsed 0.0 min, ETA 0.1 min
[2/8] p1_n64_t4_s1_pin1: 33 rows in 1s, elapsed 0.0 min, ETA 0.1 min
[3/8] p1_n256_t1_s1_pin1: 33 rows in 1s, elapsed 0.0 min, ETA 0.1 min
[4/8] p1_n256_t4_s1_pin1: 33 rows in 1s, elapsed 0.1 min, ETA 0.1 min
[5/8] p8_n64_t1_s1_pin1: 33 rows in 1s, elapsed 0.1 min, ETA 0.0 min
[6/8] p8_n64_t4_s1_pin1: 33 rows in 1s, elapsed 0.1 min, ETA 0.0 min
[7/8] p8_n256_t1_s1_pin1: 33 rows in 3s, elapsed 0.1 min, ETA 0.0 min
[8/8] p8_n256_t4_s1_pin1: 33 rows in 2s, elapsed 0.1 min, ETA 0.0 min

Raw rows -> results_quick/s03_raw_runs.csv (264 runs)
Environment -> results_quick/system_info.txt
Next: python analyze.py
264 runs, 8 configurations, correctness failures: 0 (max error 0.00e+00)
 passes  tiles  threads_req  gpu_only_ms  cpu_only_ms  best_frac  be

In [15]:
!python run_experiment.py --student-id {STUDENT_ID} --batch-id {BATCH_ID}

+ nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native /kaggle/working/s03_bench.cu -o /kaggle/working/s03_bench
18 configurations x 11 CPU shares x 7 repetitions
[1/18] p1_n256_t1_s1_pin1: 77 rows in 2s, elapsed 0.0 min, ETA 0.5 min
[2/18] p1_n256_t2_s1_pin1: 77 rows in 1s, elapsed 0.1 min, ETA 0.4 min
[3/18] p1_n256_t4_s1_pin1: 77 rows in 1s, elapsed 0.1 min, ETA 0.4 min
[4/18] p1_n1024_t1_s1_pin1: 77 rows in 5s, elapsed 0.2 min, ETA 0.6 min
[5/18] p1_n1024_t2_s1_pin1: 77 rows in 5s, elapsed 0.2 min, ETA 0.6 min
[6/18] p1_n1024_t4_s1_pin1: 77 rows in 4s, elapsed 0.3 min, ETA 0.6 min
[7/18] p1_n4096_t1_s1_pin1: 77 rows in 20s, elapsed 0.6 min, ETA 1.0 min
[8/18] p1_n4096_t2_s1_pin1: 77 rows in 16s, elapsed 0.9 min, ETA 1.1 min
[9/18] p1_n4096_t4_s1_pin1: 77 rows in 16s, elapsed 1.2 min, ETA 1.2 min
[10/18] p16_n256_t1_s1_pin1: 77 rows in 7s, elapsed 1.3 min, ETA 1.0 min
[11/18] p16_n256_t2_s1_pin1: 77 rows in 4s, elapsed 1.4 min, ETA 0.9 min
[12/18] p16_n256_t4_s1_pin

In [16]:
!python analyze.py

1386 runs, 18 configurations, correctness failures: 0 (max error 0.00e+00)
 passes  tiles  threads_req  gpu_only_ms  cpu_only_ms  best_frac  best_ms  speedup_best_vs_gpu_only  predicted_frac  hybrid_beats_gpu_only
      1    256            1       11.589       15.570        0.4    7.073                     1.638           0.427                   True
      1    256            2       11.875        8.426        0.6    5.249                     2.262           0.585                   True
      1    256            4       11.261        6.514        0.6    4.524                     2.489           0.634                   True
     16    256            1       21.353      156.383        0.1   19.216                     1.111           0.120                  False
     16    256            2       20.020       85.980        0.2   17.792                     1.125           0.189                  False
     16    256            4       20.025       78.301        0.2   16.167                  

In [17]:
!python make_report.py --name "{NAME}" --reg {REG} --batch "{BATCH_SECTION}" 

saved S03_Individual_Technical_Report.docx
abstract length: 189 words (manual asks for 150-200)


In [18]:
!zip -qr S03_submission.zip results s03_bench.cu run_experiment.py analyze.py make_report.py README.md S03_Individual_Technical_Report.docx
!ls -la S03_submission.zip S03_Individual_Technical_Report.docx

-rw-r--r-- 1 root root  378582 Oct 19 16:30 S03_Individual_Technical_Report.docx
-rw-r--r-- 1 root root 1309687 Oct 19 16:30 S03_submission.zip
